# LG Aimers 팀 후속 실험: HGB 350회 + XGBoost 고정 반복 비교

기존 전체 실행에서 확인된 과소학습 문제를 보완합니다. HGB는 350회로 고정하고, XGBoost는 25·50·100·200회로 고정해 2024 검증 정답을 early stopping에 사용하지 않습니다.

처음에는 `MODE='quick'`, `PRESET='team_next'`로 코드만 점검한 뒤 성공하면 `MODE='full'`로 변경하세요. quick 점수는 모델 채택에 사용하지 않습니다.

In [ ]:
from pathlib import Path

# ===== 사용자 설정 =====
GITHUB_USER = "tswaincae1221"
REPO_NAME = "lg_aimers_experiment_lab"  # 실제 GitHub 저장소 이름
BRANCH = "main"
REPO_IS_PRIVATE = True

DRIVE_DATA_DIR = Path("/content/drive/MyDrive/aimers_data")
DRIVE_RESULT_DIR = Path("/content/drive/MyDrive/aimers_results/experiment_lab")

MODE = "quick"       # quick 점검 후 full로 변경
PRESET = "team_next"  # HGB 350회 3종 + XGBoost 고정 반복 4종
ONLY_EXPERIMENTS = []   # 빈 목록이면 four_models 전체 실행
N_JOBS = 2
RERUN = False

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
REPO_DIR = Path("/content/lg_aimers_experiment_lab")
LOCAL_DATA_DIR = REPO_DIR / "data"
RESULT_DIR = DRIVE_RESULT_DIR / MODE
print("저장소:", REPO_URL)
print("실행 모드:", MODE, "/ 프리셋:", PRESET)
print("결과 폴더:", RESULT_DIR)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 저장소 불러오기

비공개 저장소라면 GitHub Personal Access Token을 입력합니다. 입력값은 화면에 표시되지 않습니다. 정상 저장소가 이미 있으면 `git pull`로 갱신하고, 불완전한 폴더만 정리한 뒤 다시 복제합니다. 같은 셀을 재실행해도 폴더 충돌이 나지 않습니다.

In [ ]:
import base64
import getpass
import shutil
import subprocess

if not GITHUB_USER or not REPO_NAME:
    raise ValueError("GITHUB_USER와 REPO_NAME을 설정하세요.")

token = getpass.getpass("GitHub token: " ) if REPO_IS_PRIVATE else None
basic = base64.b64encode(f"x-access-token:{token}".encode()).decode() if token else None

def run_git(*git_args):
    command = ["git", *git_args]
    if basic:
        command = ["git", "-c", f"http.extraHeader=AUTHORIZATION: basic {basic}", *git_args]
    return subprocess.run(command, text=True, capture_output=True)

if (REPO_DIR / ".git").is_dir():
    result = run_git("-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH)
    action = "업데이트"
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
        print("불완전한 코랩 폴더 정리 완료")
    result = run_git("clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR))
    action = "복제"

if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError(f"GitHub 저장소 {action} 실패")
print(f"GitHub 저장소 {action} 완료:", REPO_DIR)

assert (REPO_DIR / "config/experiments.json").is_file(), "실험 설정 파일이 없습니다."

In [ ]:
import sys

install = [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt"), "-r", str(REPO_DIR / "requirements-dev.txt")]
if PRESET in {"four_models", "team_next", "compact", "all"}:
    install.extend(["-r", str(REPO_DIR / "requirements-optional.txt")])
subprocess.run(install, check=True)
print("패키지 설치 완료")

## 데이터 확인·로컬 복사

Drive의 원본은 유지하고 Colab 로컬 SSD로 복사합니다. 파일명은 정확히 `train.csv`, `test.csv`, `trackman_history.csv`여야 합니다.

In [ ]:
import shutil

required = ["train.csv", "test.csv", "trackman_history.csv"]
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
for name in required:
    source = DRIVE_DATA_DIR / name
    if not source.is_file():
        raise FileNotFoundError(f"못 찾음: {source}")
    destination = LOCAL_DATA_DIR / name
    if (not destination.is_file() or destination.stat().st_size != source.stat().st_size or destination.stat().st_mtime_ns != source.stat().st_mtime_ns):
        shutil.copy2(source, destination)
    print(f"{name}: {destination.stat().st_size / 1024**2:.1f} MB")

mapping = REPO_DIR / "resources/pitcher_trackman_mapping.csv"
assert mapping.is_file(), mapping

In [ ]:
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=REPO_DIR,
    text=True,
    capture_output=True,
)
print(test_result.stdout)
if test_result.returncode != 0:
    print(test_result.stderr)
    raise RuntimeError("단위 테스트 실패")

## 실험 설정 확인·수정

아래 셀에서 제공 피처 묶음과 모델을 확인합니다. 직접 실험을 추가하려면 `config['experiments']`에 항목을 추가하거나 기존 모델의 `params`를 수정한 뒤 셀을 실행하세요. 원본 설정 파일은 건드리지 않고 `/content/experiments_runtime.json`에 복사됩니다.

In [ ]:
import json
import pandas as pd
from IPython.display import display

config = json.loads((REPO_DIR / "config/experiments.json").read_text(encoding="utf-8"))
display(pd.DataFrame([
    {"feature_set": name, "input_view": value.get("input_view", "v1"), "stage": value.get("preprocessing_stage"), "blocks": value.get("blocks"), "drop": value.get("drop_patterns"), "description": value.get("description")}
    for name, value in config["feature_sets"].items()
]))
display(pd.DataFrame([
    {"model": name, "type": value["type"], "description": value.get("description")}
    for name, value in config["models"].items()
]))
experiment_table = pd.DataFrame(config["experiments"])
display(experiment_table[[column for column in ["name", "feature_set", "model", "comparison_experiment", "preprocessing_comparison_experiment", "presets"] if column in experiment_table.columns]])

# 예시: learning rate 수정
# config["models"]["lgbm_base"]["params"]["learning_rate"] = 0.02
# config["models"]["lgbm_base"]["params"]["num_leaves"] = 47

RUNTIME_CONFIG = Path("/content/experiments_runtime.json")
RUNTIME_CONFIG.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
print("실행 설정 저장:", RUNTIME_CONFIG)

## 실험 실행

실행 로그를 그대로 출력합니다. 한 조합이 실패해도 오류가 해당 실행 폴더에 기록되고 다음 조합으로 넘어갑니다. 동일 데이터·동일 설정의 성공 실험은 자동으로 건너뜁니다.

In [ ]:
RESULT_DIR.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, "-u", "-m", "src.experiment_runner",
    "--config", str(RUNTIME_CONFIG),
    "--train", "data/train.csv",
    "--test", "data/test.csv",
    "--trackman", "data/trackman_history.csv",
    "--mapping", "resources/pitcher_trackman_mapping.csv",
    "--output-dir", str(RESULT_DIR),
    "--mode", MODE,
    "--preset", PRESET,
    "--validation-season", "2024",
    "--n-jobs", str(N_JOBS),
]
if ONLY_EXPERIMENTS:
    command.extend(["--only", *ONLY_EXPERIMENTS])
if RERUN:
    command.append("--rerun")

process = subprocess.Popen(
    command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"실험 실행 실패 (종료 코드 {return_code}). 위 로그와 runs/*/error.txt를 확인하세요.")
print("실험 완료:", RESULT_DIR)

## 결과 확인

`brier_delta_vs_preprocessing`이 음수면 같은 모델에서 원본 최소처리보다 전체 피처 엔지니어링이 개선된 것입니다. `brier_delta_vs_comparison`이 음수면 같은 모델에서 타자 위협도 6개 추가 후 개선된 것입니다. quick 결과를 확인한 뒤 첫 설정 셀에서 `MODE='full'`로 바꾸고 설정 셀부터 다시 실행하세요.

In [ ]:
from IPython.display import Image, Markdown, display

leaderboard = pd.read_csv(RESULT_DIR / "leaderboard.csv")
display_columns = [
    "experiment", "model_family", "preprocessing_stage", "feature_count",
    "brier", "brier_delta_vs_baseline", "brier_improvement_pct",
    "preprocessing_comparison_experiment", "brier_delta_vs_preprocessing",
    "comparison_experiment", "brier_delta_vs_comparison",
    "brier_improvement_vs_comparison_pct",
    "auc", "logloss", "ece_10bin", "elapsed_seconds"
]
display(leaderboard[[column for column in display_columns if column in leaderboard.columns]])

final_path = RESULT_DIR / "four_model_final_comparison.csv"
if final_path.is_file():
    print("최종 피처 조건 4개 모델 순위")
    display(pd.read_csv(final_path)[["model_family", "brier", "auc", "logloss", "ece_10bin", "best_iteration", "elapsed_seconds"]])

for image_name in ["four_model_final_comparison.png", "preprocessing_before_after.png", "improvement_vs_comparison.png", "leaderboard_brier.png", "score_vs_time.png"]:
    path = RESULT_DIR / image_name
    if path.is_file():
        display(Image(filename=str(path)))

report_path = RESULT_DIR / "experiment_report.md"
if report_path.is_file():
    display(Markdown(report_path.read_text(encoding="utf-8")))

## 다음 실험 추가 방법

1. `quick / four_models`로 12개 조합 코드 점검
2. `MODE='full'`, `PRESET='four_models'`로 정식 비교
3. `four_model_final_comparison.csv`에서 최종 4개 모델 순위 확인
4. `brier_delta_vs_preprocessing`과 `brier_delta_vs_comparison`으로 전처리·위협도 효과 확인
5. 점수 차이가 작으면 저장된 2024 예측값으로 경기 단위 bootstrap 검증

결과는 `experiment_history.csv`에 누적되고, 각 실행의 예측값·calibration·중요도·오류는 `runs/` 아래에 보존됩니다.